<a href="https://colab.research.google.com/github/KSamar33/samar-codeboosters-2026/blob/main/Day%207/Day7_miniproject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install chromadb sentence-transformers -q
print("Installation complete")
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
print("All libraries imported successfully")
print(f"ChromaDB version:{chromadb.__version__}")
client = chromadb.EphemeralClient()
collection = client.get_or_create_collection(name="my_collection")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 110.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 78.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the 

In [2]:
notes_df = pd.read_csv("college_notes.csv")
print(f"\nDataset loaded. Shape: {notes_df.shape}, Columns: {list(notes_df.columns)}")


Dataset loaded. Shape: (15, 4), Columns: ['note_id', 'subject', 'topic', 'content']


In [3]:
all_documents = notes_df['content'].tolist()
all_ids = notes_df['note_id'].tolist()
all_metadatas = [{
    "subject": row['subject'],
    "topic": row['topic']
} for _, row in notes_df.iterrows()]
print(f"Documents prepared: {len(all_documents)}")
print(f"IDs prepared: {len(all_ids)}")
print(f"Metadata prepared: {len(all_metadatas)}")

Documents prepared: 15
IDs prepared: 15
Metadata prepared: 15


In [4]:
print("\nLoading embedding model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"Model loaded. Produces vectors of size: {model.get_sentence_embedding_dimension()} dimensions")


Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded. Produces vectors of size: 384 dimensions


/tmp/ipykernel_2021/1181650735.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded. Produces vectors of size: {model.get_sentence_embedding_dimension()} dimensions")


In [5]:
print("\nInitializing ChromaDB client with Cosine Similarity...")
chroma_client = chromadb.Client()
collection_name = "college_notes_cosine"
try:
    chroma_client.delete_collection(name=collection_name)
except:
    pass
collection = chroma_client.create_collection(name=collection_name, metadata={"hnsw:space": "cosine"})
print(f"Collection '{collection_name}' created with cosine distance.")


Initializing ChromaDB client with Cosine Similarity...
Collection 'college_notes_cosine' created with cosine distance.


In [6]:
print("\nAdding all notes to ChromaDB collection...")
collection.add(
    documents=all_documents,
    ids=all_ids,
    metadatas=all_metadatas
)
print(f"Documents added to Collection. Total: {collection.count()}")


Adding all notes to ChromaDB collection...


/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:07<00:00, 10.6MiB/s]


Documents added to Collection. Total: 15


In [7]:
collection.add(documents=all_documents, ids=all_ids, metadatas=all_metadatas)
filtered_query_text = "Model evaluation metrics like accuracy for classification performance"
filtered_query_embedding = model.encode(filtered_query_text).tolist()
print(f"\n--- Performing Filtered Semantic Search (Cosine) for: '{filtered_query_text}' ---")
filtered_results = collection.query(query_embeddings=[filtered_query_embedding], n_results=2, where={"subject": "Machine Learning"}, include=['documents', 'distances', 'metadatas'])
for rank, (doc, doc_id, doc_dist, doc_meta) in enumerate(zip(filtered_results['documents'][0], filtered_results['ids'][0], filtered_results['distances'][0], filtered_results['metadatas'][0])):
    print(f"Rank: {rank} | ID: {doc_id} | Distance: {doc_dist:.4f}")
    print(f"Subject: {doc_meta['subject']} | Topic: {doc_meta['topic']}")
    print(f"Document: {doc[:150]}...")
    print('-' * 30)


--- Performing Filtered Semantic Search (Cosine) for: 'Model evaluation metrics like accuracy for classification performance' ---
Rank: 0 | ID: N007 | Distance: 0.2212
Subject: Machine Learning | Topic: Model Evaluation
Document: Model evaluation measures how well a machine learning model performs. Common metrics include accuracy for classification and Mean Absolute Error and R...
------------------------------
Rank: 1 | ID: N009 | Distance: 0.6834
Subject: Machine Learning | Topic: Decision Trees
Document: A decision tree is a machine learning model that makes predictions by asking a series of yes or no questions about the features. It splits data at eac...
------------------------------
